In [19]:
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets
from torchvision.transforms import v2
from torchvision.io import decode_image

import pandas as pd
import os


In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f'Device {device}')

Device cuda


In [21]:
images = pd.read_parquet('./data/img.parquet')
annot = pd.read_parquet('./data/annot.parquet')

In [22]:
images.head()

,id,width,height,set,file_name
0,a4ea732cd3d5948a,840,1024,train,train/a4ea732cd3d5948a.jpg
1,4bf43a7b2a898044,1024,683,train,train/4bf43a7b2a898044.jpg
2,1b55b309b0f50d02,1024,683,train,train/1b55b309b0f50d02.jpg
3,00c359f294f7dcd9,1024,680,train,train/00c359f294f7dcd9.jpg
4,04b5a37f762b0f51,768,1024,train,train/04b5a37f762b0f51.jpg


In [23]:
annot.head()

,id,image_id,bbox,utf8_string,points,area
0,a4ea732cd3d5948a_1,a4ea732cd3d5948a,"[525.83, 3.4, 197.64, 33.94]",Performance,"[525.83, 3.4, 723.47, 7.29, 722.76, 36.99, 525...",6707.90
1,a4ea732cd3d5948a_2,a4ea732cd3d5948a,"[534.67, 64.68, 91.22, 38.19]",Sport,"[535.73, 64.68, 623.41, 67.51, 625.89, 102.87,...",3483.69
2,a4ea732cd3d5948a_3,a4ea732cd3d5948a,"[626.95, 63.62, 96.52, 31.82]",Watch,"[626.95, 63.62, 721.7, 63.62, 723.47, 95.44, 6...",3071.27
3,a4ea732cd3d5948a_4,a4ea732cd3d5948a,"[577.4, 141.87, 147.13, 43.1]",...period.,"[580.02, 143.61, 724.53, 141.87, 723.66, 184.9...",6341.30
4,a4ea732cd3d5948a_5,a4ea732cd3d5948a,"[391.03, 163.9, 60.82, 38.65]",.,"[395.2, 163.9, 451.85, 191.94, 445.59, 202.55,...",2350.69


In [24]:
print(f'images shape {images.shape}')
print(f'annot shape {annot.shape}')

images shape (21778, 5)
annot shape (1052354, 6)


In [58]:
class ImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_dir = img_dir

        self.img_labels = pd.read_parquet(annotations_file)
        self.grouped_labels = self.img_labels.groupby('image_id')['utf8_string'].apply(list).reset_index()

        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        row = self.grouped_labels.iloc[idx]

        print(row)
        
        img_path = os.path.join(self.img_dir, f"{row['image_id']}.jpg")
        image = decode_image(img_path)
        
        labels = row['utf8_string']  # List of all labels for this image
        
        if self.transform:
            image = self.transform(image)
        
        return image, labels  # Returns: (image, ['label1', 'label2', ...])

In [59]:
dataset = ImageDataset('./data/annot.parquet', './data/images')

In [63]:
image, label  = dataset[140]

# print(image, label)

image_id                                        0038c797a3708652
utf8_string    [WIS, global, public, relation, bicon, PROJECT...
Name: 140, dtype: object
